# 在线购物评论数据探索性分析

数据集：`online_shopping_10_cats.csv`，包含10个商品类别的购物评论及情感标签。

分析目标：
- 了解数据基本结构与分布
- 分析各类别评论数量与情感倾向
- 探索评论文本长度特征
- 挖掘高频词汇，对比各类别语言特征
- 检查数据质量

## 1. 导入依赖库

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import jieba
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体，优先使用系统可用字体
import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'sans-serif']
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'sans-serif']
else:
    plt.rcParams['font.family'] = ['WenQuanYi Micro Hei', 'DejaVu Sans', 'sans-serif']

plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
plt.rcParams['figure.dpi'] = 100

print('依赖库加载完毕')
print(f'pandas  版本: {pd.__version__}')
print(f'matplotlib 版本: {matplotlib.__version__}')
print(f'seaborn 版本: {sns.__version__}')
print(f'jieba   版本: {jieba.__version__}')

## 2. 加载数据

In [ ]:
DATA_PATH = '../data/online_shopping_10_cats.csv'

df = pd.read_csv(DATA_PATH, encoding='utf-8')

print('=' * 50)
print(f'数据集形状: {df.shape}  (行数, 列数)')
print('=' * 50)
print('\n列类型:')
print(df.dtypes)
print('\n前5行数据:')
df.head()

In [ ]:
print('基本统计描述:')
df.describe(include='all')

## 3. 类别分布（cat 列）

In [ ]:
cat_counts = df['cat'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))

colors = sns.color_palette('tab10', n_colors=len(cat_counts))
bars = ax.bar(cat_counts.index, cat_counts.values, color=colors, edgecolor='white', linewidth=0.8)

# 在柱顶标注数量
for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f'{val:,}', ha='center', va='bottom', fontsize=9)

ax.set_title('各商品类别评论数量分布', fontsize=15, fontweight='bold', pad=12)
ax.set_xlabel('商品类别', fontsize=12)
ax.set_ylabel('评论数量', fontsize=12)
ax.set_ylim(0, cat_counts.max() * 1.12)
ax.tick_params(axis='x', labelsize=11)
sns.despine()
plt.tight_layout()
plt.savefig('../results/cat_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n各类别数量:')
print(cat_counts.to_frame('数量').assign(占比=lambda x: (x['数量'] / len(df) * 100).round(2).astype(str) + '%'))

## 4. 正负情感分布（label 列）

In [ ]:
label_map = {1: '正面', 0: '负面'}
df['label_name'] = df['label'].map(label_map)
label_counts = df['label_name'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左：整体情感饼图
palette = {'正面': '#4CAF50', '负面': '#F44336'}
wedge_colors = [palette[k] for k in label_counts.index]
axes[0].pie(
    label_counts.values,
    labels=label_counts.index,
    colors=wedge_colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops={'fontsize': 12}
)
axes[0].set_title('整体情感分布', fontsize=13, fontweight='bold')

# 右：各类别正负比例堆叠柱状图
cat_label = df.groupby(['cat', 'label_name']).size().unstack(fill_value=0)
# 按正面数量降序排列
cat_label = cat_label.loc[cat_label.sum(axis=1).sort_values(ascending=False).index]
cat_label_pct = cat_label.div(cat_label.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(cat_label_pct))
for col in ['正面', '负面']:
    if col in cat_label_pct.columns:
        axes[1].bar(cat_label_pct.index, cat_label_pct[col],
                    bottom=bottom, label=col, color=palette[col],
                    edgecolor='white', linewidth=0.5)
        bottom += cat_label_pct[col].values

axes[1].set_title('各类别情感比例（%）', fontsize=13, fontweight='bold')
axes[1].set_xlabel('商品类别', fontsize=11)
axes[1].set_ylabel('比例 (%)', fontsize=11)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].set_ylim(0, 110)
axes[1].tick_params(axis='x', labelsize=10)
sns.despine(ax=axes[1])

plt.suptitle('情感标签分布分析', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/label_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print('情感标签计数:')
print(label_counts)

## 5. 评论文本长度分布

In [ ]:
df['review_len'] = df['review'].astype(str).apply(len)

print('评论长度统计:')
print(df['review_len'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：全量分布直方图（截断到95分位数，避免极端值影响可视化）
q95 = df['review_len'].quantile(0.95)
data_clipped = df.loc[df['review_len'] <= q95, 'review_len']

axes[0].hist(data_clipped, bins=60, color='steelblue', edgecolor='white', linewidth=0.4, alpha=0.85)
axes[0].axvline(df['review_len'].mean(), color='red', linestyle='--', linewidth=1.5, label=f'均值 {df["review_len"].mean():.0f}')
axes[0].axvline(df['review_len'].median(), color='orange', linestyle='--', linewidth=1.5, label=f'中位数 {df["review_len"].median():.0f}')
axes[0].set_title(f'评论长度分布（截断至95分位={q95:.0f}字符）', fontsize=12, fontweight='bold')
axes[0].set_xlabel('评论字符数', fontsize=11)
axes[0].set_ylabel('频次', fontsize=11)
axes[0].legend(fontsize=10)
sns.despine(ax=axes[0])

# 右：正负评论长度对比KDE
for lname, color in [('正面', '#4CAF50'), ('负面', '#F44336')]:
    subset = df.loc[(df['label_name'] == lname) & (df['review_len'] <= q95), 'review_len']
    axes[1].hist(subset, bins=50, alpha=0.5, color=color, label=lname, edgecolor='none', density=True)

axes[1].set_title('正面 vs 负面评论长度密度分布', fontsize=12, fontweight='bold')
axes[1].set_xlabel('评论字符数', fontsize=11)
axes[1].set_ylabel('密度', fontsize=11)
axes[1].legend(fontsize=10)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('../results/review_length_dist.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. 各类别评论长度箱线图

In [ ]:
# 按各类别中位长度排序
cat_order = df.groupby('cat')['review_len'].median().sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))

sns.boxplot(
    data=df[df['review_len'] <= df['review_len'].quantile(0.99)],
    x='cat', y='review_len',
    order=cat_order,
    palette='tab10',
    flierprops=dict(marker='o', markersize=2, alpha=0.3),
    ax=ax
)

ax.set_title('各商品类别评论长度箱线图（截断至99分位）', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('商品类别', fontsize=12)
ax.set_ylabel('评论字符数', fontsize=12)
ax.tick_params(axis='x', labelsize=11)
sns.despine()
plt.tight_layout()
plt.savefig('../results/review_length_boxplot.png', bbox_inches='tight', dpi=150)
plt.show()

print('各类别评论长度统计（中位数排序）:')
df.groupby('cat')['review_len'].agg(['mean', 'median', 'std', 'min', 'max']).loc[cat_order].round(1)

## 7. 词频分析：整体 Top-30 高频词

In [ ]:
# 停用词（常见无意义词）
STOPWORDS = set([
    '的', '了', '是', '在', '我', '有', '和', '就', '不', '人', '都', '一', '一个',
    '上', '也', '很', '到', '说', '要', '去', '你', '会', '着', '没有', '看', '好',
    '来', '这', '他', '她', '它', '我们', '你们', '他们', '这个', '那个', '但', '但是',
    '所以', '因为', '如果', '虽然', '还是', '已经', '可以', '没', '还', '就是', '非常',
    '这样', '那么', '只', '被', '让', '把', '用', '又', '对', '比', '跟', '给',
    '自己', '什么', '这种', '真的', '感觉', '觉得', '以为', '认为', '一些', '一点',
    '一直', '一下', '两个', '东西', '时候', '时间', '地方', '方面', '情况', '问题',
    '而且', '不是', '因此', '其实', '不过', '然后', '之后', '之前', '其他', '不能',
    '不会', '不要', '大', '小', '多', '少', '好的', '没有', '这里', '那里', '而'
])

print('正在对所有评论进行jieba分词，请稍候...')
all_words = []
for text in df['review'].astype(str):
    words = jieba.cut(text)
    for w in words:
        w = w.strip()
        if len(w) >= 2 and w not in STOPWORDS:
            all_words.append(w)

word_counter = Counter(all_words)
top30 = word_counter.most_common(30)
top30_words, top30_freqs = zip(*top30)

print(f'\n分词完成，共提取有效词语 {len(all_words):,} 个，词汇量 {len(word_counter):,}')

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, 30))
bars = ax.bar(range(30), top30_freqs, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(30))
ax.set_xticklabels(top30_words, rotation=45, ha='right', fontsize=10)
ax.set_title('全量评论 Top-30 高频词', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('词语', fontsize=12)
ax.set_ylabel('出现频次', fontsize=12)

for bar, freq in zip(bars, top30_freqs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f'{freq:,}', ha='center', va='bottom', fontsize=7, rotation=90)

sns.despine()
plt.tight_layout()
plt.savefig('../results/top30_words.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. 各类别高频词对比

In [ ]:
categories = df['cat'].unique().tolist()
TOP_N = 10  # 每个类别展示前10词

# 预先分词并按类别存储
cat_word_dict = {}
for cat in categories:
    cat_texts = df.loc[df['cat'] == cat, 'review'].astype(str)
    words = []
    for text in cat_texts:
        for w in jieba.cut(text):
            w = w.strip()
            if len(w) >= 2 and w not in STOPWORDS:
                words.append(w)
    cat_word_dict[cat] = Counter(words)

# 绘制子图：2行5列，每个类别一个横向柱状图
n_cols = 5
n_rows = int(np.ceil(len(categories) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 7 * n_rows))
axes = axes.flatten()

palette = sns.color_palette('tab10', n_colors=len(categories))

for idx, cat in enumerate(categories):
    top_words = cat_word_dict[cat].most_common(TOP_N)
    words_list = [w for w, _ in top_words[::-1]]  # 从小到大排列（水平图从上到下是大到小）
    freq_list  = [f for _, f in top_words[::-1]]

    ax = axes[idx]
    ax.barh(words_list, freq_list, color=palette[idx], edgecolor='white', linewidth=0.5)
    for i, (w, f) in enumerate(zip(words_list, freq_list)):
        ax.text(f + freq_list[-1] * 0.01, i, f'{f:,}', va='center', fontsize=8)
    ax.set_title(f'{cat}', fontsize=12, fontweight='bold')
    ax.set_xlabel('频次', fontsize=9)
    ax.tick_params(axis='y', labelsize=10)
    sns.despine(ax=ax)

# 隐藏多余子图
for j in range(len(categories), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'各类别 Top-{TOP_N} 高频词对比', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/cat_top_words.png', bbox_inches='tight', dpi=150)
plt.show()

## 9. 数据质量检查

In [ ]:
print('=' * 55)
print('  数据质量报告')
print('=' * 55)

# 1. 空值检查
print('\n[1] 各列空值统计:')
null_info = pd.DataFrame({
    '空值数量': df.isnull().sum(),
    '空值比例(%)': (df.isnull().mean() * 100).round(4)
})
print(null_info)

# 2. 重复行检查
n_dup = df.duplicated().sum()
n_dup_review = df.duplicated(subset=['review']).sum()
print(f'\n[2] 完全重复行数量: {n_dup} ({n_dup/len(df)*100:.2f}%)')
print(f'    仅review重复行数量: {n_dup_review} ({n_dup_review/len(df)*100:.2f}%)')

# 3. label值域检查
invalid_label = df.loc[~df['label'].isin([0, 1])]
print(f'\n[3] label异常值（非0/1）: {len(invalid_label)} 条')

# 4. 空评论检查
empty_review = df.loc[df['review'].astype(str).str.strip() == '']
print(f'\n[4] 空评论（空白字符串）: {len(empty_review)} 条')

# 5. 极短/极长评论
very_short = df[df['review_len'] < 5]
very_long  = df[df['review_len'] > 2000]
print(f'\n[5] 极短评论（<5字符）: {len(very_short)} 条')
print(f'    极长评论（>2000字符）: {len(very_long)} 条')

print('\n[6] 各类别×情感 交叉计数:')
cross = pd.crosstab(df['cat'], df['label_name'])
cross['总计'] = cross.sum(axis=1)
print(cross)

print('\n[7] 数据总览:')
print(f'    总条数         : {len(df):,}')
print(f'    类别数         : {df["cat"].nunique()}')
print(f'    正面评论       : {(df["label"]==1).sum():,} ({(df["label"]==1).mean()*100:.1f}%)')
print(f'    负面评论       : {(df["label"]==0).sum():,} ({(df["label"]==0).mean()*100:.1f}%)')
print(f'    平均评论长度   : {df["review_len"].mean():.1f} 字符')
print(f'    最长评论       : {df["review_len"].max()} 字符')
print(f'    最短评论       : {df["review_len"].min()} 字符')
print('=' * 55)

In [ ]:
# 数据质量可视化：空值热力图 + 重复评论分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：各类别数据完整性
completeness = (1 - df.groupby('cat').apply(lambda x: x.isnull().mean())).T
if completeness.shape[0] > 1:
    sns.heatmap(completeness, annot=True, fmt='.3f', cmap='YlGn',
                vmin=0.9, vmax=1.0, ax=axes[0],
                linewidths=0.5, cbar_kws={'label': '完整率'})
    axes[0].set_title('各类别字段完整率热力图', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('商品类别', fontsize=10)
    axes[0].tick_params(axis='x', rotation=45, labelsize=9)
else:
    axes[0].text(0.5, 0.5, '无空值，数据完整', ha='center', va='center',
                 fontsize=14, transform=axes[0].transAxes)
    axes[0].set_title('各类别字段完整率', fontsize=12, fontweight='bold')

# 右：评论长度分位数分布（各类别对比）
cat_len_stats = df.groupby('cat')['review_len'].quantile([0.25, 0.5, 0.75]).unstack()
cat_len_stats.columns = ['Q1(25%)', '中位数', 'Q3(75%)']
cat_len_stats_sorted = cat_len_stats.sort_values('中位数', ascending=False)

x = np.arange(len(cat_len_stats_sorted))
width = 0.25
for i, (col, color) in enumerate(zip(['Q1(25%)', '中位数', 'Q3(75%)'],
                                       ['#90CAF9', '#1565C0', '#0D47A1'])):
    axes[1].bar(x + i * width, cat_len_stats_sorted[col], width,
                label=col, color=color, edgecolor='white')

axes[1].set_xticks(x + width)
axes[1].set_xticklabels(cat_len_stats_sorted.index, rotation=45, ha='right', fontsize=9)
axes[1].set_title('各类别评论长度分位数对比', fontsize=12, fontweight='bold')
axes[1].set_ylabel('字符数', fontsize=10)
axes[1].legend(fontsize=9)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('../results/data_quality.png', bbox_inches='tight', dpi=150)
plt.show()